# Task 2 Predictions with SVD

This notebook creates final Task 2 recommendations for users in `data/raw/ratings_test.csv`.
It uses SVD as the main model, routes cold-start users to a fallback ranker, and writes the final submission CSV.


## Workflow

1. Load ratings and movie data.
2. Fit `SVDModel` for personalization.
3. Fit `BayesianColdStartRanker` for cold-start users.
4. Route user inference with `RecommenderInferenceRouter`.
5. Validate no seen movies are recommended.
6. Save output CSV with required columns.


In [1]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from src.models.cold_start import BayesianColdStartRanker
from src.models.inference_router import RecommenderInferenceRouter
from src.models.svd_model import SVDModel


/home/samuel/Documents/data-mining-recommender-systems/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Resolve project root from notebook location.
project_root_path = Path.cwd()
if not (project_root_path / "src").exists():
    project_root_path = project_root_path.parent

train_ratings_path = project_root_path / "data" / "processed" / "ratings_train_cleaned.csv"
if not train_ratings_path.exists():
    train_ratings_path = project_root_path / "data" / "raw" / "ratings_train.csv"

movies_features_path = project_root_path / "data" / "processed" / "movies_cleaned.csv"
if not movies_features_path.exists():
    movies_features_path = project_root_path / "data" / "raw" / "movies.csv"

ratings_test_template_path = project_root_path / "data" / "raw" / "ratings_test.csv"
output_predictions_path = project_root_path / "data" / "processed" / "ratings_test_filled.csv"

train_ratings_dataframe = pd.read_csv(train_ratings_path)
movies_dataframe = pd.read_csv(movies_features_path)
ratings_test_dataframe = pd.read_csv(ratings_test_template_path)

print(f"Train rows: {len(train_ratings_dataframe)}")
print(f"Movies rows: {len(movies_dataframe)}")
print(f"Test users: {len(ratings_test_dataframe)}")


Train rows: 97801
Movies rows: 9742
Test users: 100


In [3]:
top_n_recommendations = 10
minimum_personalization_interactions = 2

recommender_model = SVDModel()
recommender_model.fit(ratings_dataframe=train_ratings_dataframe)

cold_start_ranker = BayesianColdStartRanker()
cold_start_ranker.fit(
    ratings_dataframe=train_ratings_dataframe,
    movies_dataframe=movies_dataframe,
)

inference_router = RecommenderInferenceRouter(
    trained_model=recommender_model,
    cold_start_ranker=cold_start_ranker,
    ratings_dataframe=train_ratings_dataframe,
    minimum_personalization_interactions=minimum_personalization_interactions,
)

print("Fitted model: svd")


Fitted model: svd


In [4]:
prediction_rows = []
recommendation_source_values = []

for user_identifier in tqdm(ratings_test_dataframe["userId"].astype(int).tolist(), desc="Task2 users"):
    recommendation_result = inference_router.recommend_for_user(
        user_identifier=user_identifier,
        number_of_recommendations=top_n_recommendations,
    )

    recommendation_source_values.append(recommendation_result.source_name)
    recommended_movie_identifiers = [movie_id for movie_id, _ in recommendation_result.recommendations]

    # Keep fixed output width for submission schema.
    if len(recommended_movie_identifiers) < top_n_recommendations:
        recommended_movie_identifiers = recommended_movie_identifiers + [None] * (
            top_n_recommendations - len(recommended_movie_identifiers)
        )

    prediction_row = {"userId": int(user_identifier)}
    for recommendation_index in range(top_n_recommendations):
        column_name = f"recommendation{recommendation_index + 1}"
        prediction_row[column_name] = recommended_movie_identifiers[recommendation_index]
    prediction_rows.append(prediction_row)

predictions_dataframe = pd.DataFrame(prediction_rows)
source_counts_series = pd.Series(recommendation_source_values).value_counts()

display(predictions_dataframe.head())
display(source_counts_series)


personalized_model                              57
personalized_model_with_cold_start_injection    33
cold_start_fallback:popular_genre_coverage      10
Name: count, dtype: int64

In [5]:
seen_movies_by_user_map = (
    train_ratings_dataframe.groupby("userId")["movieId"]
    .apply(lambda movie_series: set(movie_series.astype(int)))
    .to_dict()
)

for prediction_row in predictions_dataframe.itertuples(index=False):
    user_identifier = int(prediction_row.userId)
    seen_movie_identifiers = seen_movies_by_user_map.get(user_identifier, set())
    recommended_movie_identifiers = {
        int(movie_id)
        for movie_id in prediction_row[1:]
        if pd.notna(movie_id)
    }
    overlap_movie_identifiers = seen_movie_identifiers.intersection(recommended_movie_identifiers)
    assert not overlap_movie_identifiers, (
        f"User {user_identifier} has seen items in recommendations: {sorted(overlap_movie_identifiers)}"
    )

expected_columns = ["userId"] + [f"recommendation{i}" for i in range(1, top_n_recommendations + 1)]
assert list(predictions_dataframe.columns) == expected_columns, "Column mismatch in output dataframe."
assert len(predictions_dataframe) == len(ratings_test_dataframe), "Row count mismatch with ratings_test.csv."

print("Validation checks passed.")


Validation checks passed.


In [6]:
output_predictions_path.parent.mkdir(parents=True, exist_ok=True)
predictions_dataframe.to_csv(output_predictions_path, index=False)

print(f"Saved predictions to: {output_predictions_path}")
print(f"Output rows: {len(predictions_dataframe)}")
print("Output columns:", list(predictions_dataframe.columns))


Saved predictions to: /home/samuel/Documents/data-mining-recommender-systems/data/processed/ratings_test_filled.csv
Output rows: 100
Output columns: ['userId', 'recommendation1', 'recommendation2', 'recommendation3', 'recommendation4', 'recommendation5', 'recommendation6', 'recommendation7', 'recommendation8', 'recommendation9', 'recommendation10']
